In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.10 The Ising Model: Emergence, Symmetry Breaking, and Universality

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.10",
    title="The Ising Model: Emergence, Symmetry Breaking, and Universality",
    blurb="Where order is born. From the simplest local rule — neighbouring "
    "spins prefer to align — a sharp phase transition emerges: below a critical "
    "temperature the system spontaneously magnetizes, choosing one of two equal "
    "directions. We watch symmetry break, see why a finite system never quite "
    "breaks it, and meet the astonishing fact that wholly different systems "
    "behave identically near their critical points. The quantitative machinery "
    "lives in Materials Modelling; here we are after what it all means.",
    difficulty="advanced",
    estimate="150–190 min",
)

## Notebook overview

This is the conceptual climax of the volume. Everything we built — counting microstates ([§5.1](counting.ipynb)), the
Boltzmann weight ([§5.4](microstates-entropy-temperature.ipynb)), the Metropolis algorithm ([§5.8](partition-function.ipynb)), the fluctuation–response relations ([§5.9](grand-canonical-ensemble-equivalence.ipynb)) —
converges here on a single, almost embarrassingly simple model, and out of it comes something none
of the parts contained: a sharp **phase transition**, and the spontaneous birth of order from a law
that has no preferred direction. The Ising model is a grid of spins that each point up or down and
prefer to agree with their neighbours. That is the entire rule. Yet below a critical temperature
the grid suddenly *chooses* — it magnetizes, picking one of two perfectly equivalent directions —
and this collective decision, made by no single spin, is **emergence** in its cleanest form.

We are after the *meaning* of this, not the numbers. There is a magnificent quantitative theory of
the Ising model — Onsager's exact solution, the transfer matrix, finite-size scaling, the critical
exponents, cluster algorithms — and it is developed in full in the **Materials Modelling (MMM)
course**, where Ising is the vehicle for learning Monte Carlo because it can be solved exactly and
checked against. We do not reproduce that here; we cross-reference it, and instead spend our effort
on three ideas that the quantitative treatment rushes past:

- **Spontaneous symmetry breaking.** The Hamiltonian is perfectly symmetric — flip every spin and
  the energy is unchanged — yet below $T_c$ the equilibrium state is *not* symmetric. The law
  respects a symmetry the state violates.
- **Ergodicity breaking** — the deep point, and the novel core of this notebook. A *finite* Ising
  magnet does not truly break the symmetry: given long enough, its magnetization flips sign. The
  familiar plots of $|m|$ quietly hide this. True symmetry breaking — a permanently chosen
  direction — exists only in the limit $N\to\infty$, and it happens by **ergodicity breaking**, the
  exact counterpart to the ergodicity we studied in [§5.5](ergodicity.ipynb).
- **Universality.** The very same model is a magnet, a fluid at its critical point, a binary alloy —
  unrelated systems that share *identical* critical behaviour, because near a critical point the
  microscopic details wash out (Ising-type spins reach even further, to neural networks). It is why a
  toy explains reality.

The Metropolis algorithm of [§5.8](partition-function.ipynb) is our microscope: we **invoke** it (we do not rebuild it) to watch
the lattice order, to track the telltale sign-flipping, and to measure the susceptibility — which,
as [§5.9](grand-canonical-ensemble-equivalence.ipynb) promised, is a fluctuation that *diverges* at the critical point. By the end the whole arc of
the volume will have closed: we began by counting microstates, and we end by watching order create
itself.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the lattice orders below $T_c$ and disorders above; the symmetric law yields a
> non-symmetric ordered state; a finite system's magnetization flips sign, less and less often as
> it grows; the susceptibility (a [§5.9](grand-canonical-ensemble-equivalence.ipynb) fluctuation) peaks at $T_c$; the magnet-to-fluid mapping is
> exact; and a quench freezes in spontaneous order. A ✓ is strong evidence; a ✗ is a prompt to
> *locate the discrepancy*.
>
> **Scope, and the bidirectional MMM link.** This notebook is about *concepts* — emergence,
> symmetry breaking, ergodicity breaking, universality. **All quantitative apparatus** (the exact
> 1-D transfer matrix, Onsager's 2-D solution, finite-size scaling, the critical exponents, the
> correlation length $\xi(T)$, cluster algorithms, the Binder cumulant) lives in the **Materials
> Modelling course** and is referenced there, not derived here. The renormalization-group
> *explanation* of universality is the horizon beyond both. See Sethna, *Statistical Mechanics*;
> Goldenfeld, *Lectures on Phase Transitions*; and Notebooks [§5.5](ergodicity.ipynb) (ergodicity), [§5.8](partition-function.ipynb) (Metropolis),
> [§5.9](grand-canonical-ensemble-equivalence.ipynb) (the susceptibility fluctuation, the criticality bridge), [§5.1](counting.ipynb)/[§5.4](microstates-entropy-temperature.ipynb) (counting/Boltzmann).

## Theory in brief

### The model and its symmetry

Spins $\sigma_i=\pm1$ sit on a lattice and prefer to align with their neighbours,

```{math}
:label: eq-ising
E=-J\sum_{\langle ij\rangle}\sigma_i\sigma_j\qquad(J>0,\ \text{nearest neighbours}) ,
```

the simplest model of cooperation (MMM develops it fully). The one structural fact we need: $E$ has
a $\mathbb{Z}_2$ **symmetry** — flip *every* spin, $\sigma_i\to-\sigma_i$, and the energy is
unchanged — so the two fully ordered states (all up, all down) are exactly degenerate. Whether the
system *respects* that symmetry is the question of the notebook.

### Spontaneous symmetry breaking

Below a critical temperature $T_c$ the system **spontaneously magnetizes**: it picks one of the two
degenerate ordered states, and the symmetric $m=0$ state becomes unstable,

```{math}
:label: eq-ssb
m=\frac1N\sum_i\sigma_i\ \to\ \pm m_0\ (T<T_c),\qquad m\to0\ (T>T_c) .
```

This is **spontaneous symmetry breaking**: the *law* (the Hamiltonian) is symmetric, but the
equilibrium *state* is not. Above $T_c$ thermal agitation wins and $m=0$. The order parameter $m$
tells the phases apart. The Onsager benchmarks — $T_c=2/\ln(1+\sqrt2)\approx2.269\,J/k$ and the
exact magnetization with its $\tfrac18$ exponent — are *known results*, derived in MMM/Onsager, not
here.

### Ergodicity breaking — the deep point

Here is what a quick treatment hides. A *finite* Ising system below $T_c$ does **not** truly break
the symmetry: given enough time its magnetization **flips sign**, tunnelling between the two ordered
states through a disordered fluctuation, so the long-time average of the *signed* $m$ is zero — the
symmetry is respected after all,

```{math}
:label: eq-ergodicity-breaking
\langle m\rangle_{\text{time}}=0\ \text{for finite }N;\qquad \text{flip time}\sim e^{cL^{d-1}}\to\infty\ (N\to\infty) .
```

We plot $|m|$ precisely to *mask* this finite-size flipping. True symmetry breaking — a permanently
chosen sign — emerges only as $N\to\infty$, where the free-energy barrier between the two states
(proportional to the domain-wall area) makes the flip time diverge. The system then ceases to
explore all of its accessible states — it is trapped in one symmetry sector. This is **ergodicity
breaking**, the exact counterpart to the ergodicity of [§5.5](ergodicity.ipynb), and it is the mechanism behind *every*
phase transition: how a symmetry that holds for every finite system can fail in the limit.

### Criticality, fluctuations, and the [§5.9](grand-canonical-ensemble-equivalence.ipynb) bridge

Near $T_c$ the magnetization fluctuations become enormous. The susceptibility is their measure — the
fluctuation–response relation of [§5.9](grand-canonical-ensemble-equivalence.ipynb) —

```{math}
:label: eq-ising-criticality
\chi=\frac{N}{kT}\operatorname{Var}(|m|)\ \xrightarrow{T\to T_c}\ \text{peak (and, as }N\to\infty,\ \infty) .
```

This is exactly the breakdown [§5.9](grand-canonical-ensemble-equivalence.ipynb) foretold: at a critical point the fluctuations do **not** vanish
as $1/\sqrt N$, the response diverges, and the system becomes correlated over *all* length scales —
the correlation length $\xi$ diverges, and domains of every size appear at once (critical
opalescence). We show this conceptually and visually; the quantitative $\xi(T)$, the exponent $\nu$,
and finite-size scaling are MMM's.

### Universality

Because the correlation length is huge at criticality, the system looks the same at every scale
(**scale invariance**), and the microscopic details stop mattering — only the dimensionality and
the symmetry survive,

```{math}
:label: eq-universality
\sigma_i=\pm1\ \longleftrightarrow\ n_i=\tfrac{1+\sigma_i}{2}\in\{0,1\}\quad(\text{magnet}\leftrightarrow\text{lattice gas}) .
```

So **wildly different systems share identical critical behaviour**: the Ising magnet, the lattice
**gas** (the liquid–vapour critical point), and a binary **alloy** (order–disorder) fall into the
*same universality class* — and the same Ising-type spins reach even further, into
associative-memory neural-network models. A magnet losing its magnetization and a fluid at its
critical point are, near criticality, the *same model*. Universality is why a
stripped-down toy explains real, unrelated experiments — the deepest idea in the theory of phase
transitions. (Its explanation, the renormalization group, is the horizon beyond both courses.)

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"
SPIN_CMAP = ListedColormap([INK, ACCENT])  # down = ink, up = amber

T_C = 2.0 / np.log(
    1.0 + np.sqrt(2.0)
)  # Onsager's exact 2-D critical temperature (J = k = 1)


def ising_sweep(s, beta, rng, masks):
    """One Metropolis sweep of the 2-D Ising lattice, in place (the algorithm of §5.8).

    Uses a checkerboard decomposition: the two colours of a square lattice have no same-colour
    neighbours, so every site of one colour may be updated *simultaneously* and the sweep is exact
    while vectorizing over the whole sublattice. The neighbour sum is formed with `numpy.roll`
    (periodic wrap, equivalent to indexing modulo L), and each spin is flipped with the Metropolis
    acceptance ``rng.random(...) < np.exp(-beta*dE)`` of Notebook §5.8, where the energy change to
    flip σ_i is ΔE = 2J·σ_i·Σ_nbr σ_j.

    Parameters
    ----------
    s : numpy.ndarray
        The L×L spin lattice (values ±1); modified in place.
    beta : float
        Inverse temperature 1/kT.
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``).
    masks : tuple of numpy.ndarray
        The two checkerboard boolean masks.

    Returns
    -------
    None
        Updates ``s`` in place.
    """
    for mask in masks:
        nbr = (
            np.roll(s, 1, 0) + np.roll(s, -1, 0) + np.roll(s, 1, 1) + np.roll(s, -1, 1)
        )
        dE = 2.0 * s * nbr  # ΔE to flip each spin (J = 1)
        flip = (
            rng.random(s.shape) < np.exp(-beta * dE)
        ) & mask  # Metropolis acceptance
        s[flip] *= -1


def ising_metropolis(L, T, n_sweeps, rng, init=None, burn=0):
    """Simulate the 2-D Ising model by Metropolis (Notebook §5.8) and return its magnetization trace.

    Parameters
    ----------
    L : int
        Linear lattice size (L×L spins, periodic).
    T : float
        Temperature (units J/k, with J = k = 1).
    n_sweeps : int
        Number of Metropolis sweeps.
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``).
    init : int or None, optional
        Initial condition: ``None`` for random, or ``+1``/``-1`` for a fully ordered start.
    burn : int, optional
        Sweeps to discard before recording the magnetization (default ``0``).

    Returns
    -------
    tuple
        ``(m_trace, config)`` — the per-sweep magnetization m = (1/N)·Σ_i σ_i (after
        burn-in) and the final spin configuration.
    """
    beta = 1.0 / T
    if init in (1, -1):
        s = np.full((L, L), init, dtype=int)  # fully ordered start
    else:
        s = rng.choice(np.array([-1, 1]), size=(L, L))  # random/disordered start
    ii, jj = np.indices((L, L))
    masks = ((ii + jj) % 2 == 0, (ii + jj) % 2 == 1)
    m_trace = []
    for sweep in range(n_sweeps):
        ising_sweep(s, beta, rng, masks)
        if sweep >= burn:
            m_trace.append(s.mean())
    return np.array(m_trace), s

## Exercise 1 — The model, its symmetry, and a first look at the transition

We begin compactly, because the model and its mechanics belong to MMM; what we need is the
*picture*. The rule is {eq}`eq-ising`: spins on a grid, each $\pm1$, with an energy that is lowered
whenever neighbours agree. The single structural fact that will drive everything is the
$\mathbb{Z}_2$ symmetry — flipping all spins together costs no energy, so "all up" and "all down"
are exactly equivalent. Now run the Metropolis algorithm of [§5.8](partition-function.ipynb) at two temperatures and look
({numref}`fig-ising-snapshots`). Hot, the lattice is a structureless salt-and-pepper of up and down,
with magnetization $m\approx0$: thermal agitation overwhelms the alignment preference. Cold, the
lattice has organized itself into large domains of a single sign, with $|m|$ near one: the alignment
preference has won. Somewhere between sits the **critical temperature**, $T_c=2/\ln(1+\sqrt2)\approx
2.269$ (Onsager's exact result, derived in MMM), where the change from disorder to order is sharp.

**This exercise (worked).** Run `ising_metropolis` on a lattice at a low ($T=1.6$) and a high
($T=3.5$) temperature, display the two configurations, and confirm $|m|$ is large below $T_c$ and
near zero above.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    mag_cold > 0.7 and mag_hot < 0.2,
    "the system orders below T_c (|m| large) and disorders above it (|m| ≈ 0)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Spontaneous symmetry breaking

Now we make the central concept precise. The Hamiltonian {eq}`eq-ising` is unchanged when we flip
every spin — it has no preference between up and down — and yet below $T_c$ the system ends up
magnetized in *one* direction. How can a symmetric law produce an unsymmetric outcome? This is
**spontaneous symmetry breaking** {eq}`eq-ssb`, and the word "spontaneous" is the point: nothing in
the rules picks the direction, the system picks it itself, and which way it goes is decided by
history (the initial condition, or the chance fluctuations as it cools). We can see the degeneracy
directly: start the same cold system from "all up" and from "all down" and it settles into two
mirror-image ordered states, $+m_0$ and $-m_0$, both equally valid solutions of the same symmetric
problem ({numref}`fig-ising-ssb`). Above $T_c$ there is no such choice to make — the unique
equilibrium state is the symmetric, disordered one with $m=0$. The broken symmetry below $T_c$ is
not in the laws; it is in the *state* the laws permit the system to fall into.

**This exercise (worked).** Run the cold system from an all-up start and from an all-down start, and
confirm the two settle into oppositely-magnetized ordered states — the same energy, the same
physics, opposite choices.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    mag_up > 0.7 and mag_down < -0.7,
    "the symmetric Hamiltonian yields a non-symmetric ordered state whose sign is set by history — spontaneous symmetry breaking",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Ergodicity breaking: why a finite system never quite breaks the symmetry

Here is the novel heart of the notebook, and the place where the honest story diverges from the
tidy one. We just saw a finite cold lattice "choose" a magnetization sign — but watch it long
enough and something disquieting happens: the sign **flips**. The whole magnet, fluctuation by
fluctuation, tunnels through a disordered configuration and re-orders the other way, then back
again ({numref}`fig-ising-flipping`). Over a long run the *signed* magnetization spends equal time
at $+m_0$ and $-m_0$, so its time average is **zero** {eq}`eq-ergodicity-breaking` — the symmetry is
respected after all. A finite Ising magnet does *not* truly break the symmetry. The plots of $|m|$
that everyone shows, ours included, take the absolute value precisely to hide this flipping; they
are a kindly lie. The signed magnetization tells the truth.

What rescues genuine symmetry breaking is *size*. To flip, the system must build a domain wall
across itself, and the free-energy cost of that wall grows with the lattice (its area), so the time
between flips grows explosively with $N$ — and diverges as $N\to\infty$. A macroscopic magnet would
take longer than the age of the universe to flip, so it is, for all purposes, permanently stuck in
the sector it chose. That trapping is **ergodicity breaking**: the system stops exploring all of
its accessible states, the exact counterpart to the ergodicity of [§5.5](ergodicity.ipynb). It is *how* a symmetry that
holds for every finite system fails in the limit — the mechanism of every phase transition.

**This exercise (worked).** Track the *signed* magnetization $m(t)$ for small lattices below $T_c$
over a long run, count the sign flips (a hysteresis test on $m$), and show the flip count collapses
as the lattice grows — flipping freely for $L=6$, rarely for $L=16$. The symmetry is broken only in
the thermodynamic limit.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    flip_counts[6] > flip_counts[16]
    and flip_counts[6] >= 5
    and flip_counts[16] <= flip_counts[6] // 3,
    "a finite Ising magnet's magnetization flips sign over time, less and less often as it grows — symmetry is broken only as N→∞ (ergodicity breaking)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Criticality and diverging fluctuations

We now connect the transition to the fluctuation machinery of [§5.9](grand-canonical-ensemble-equivalence.ipynb), and find that the bridge it
built carries us exactly here. Near $T_c$ the magnetization does not merely change its average — it
*fluctuates* violently, as patches of order form and dissolve on every scale at once. The measure
of those fluctuations is the **susceptibility**, and by the fluctuation–response relation of [§5.9](grand-canonical-ensemble-equivalence.ipynb) it
is $\chi=N\operatorname{Var}(|m|)/kT$ {eq}`eq-ising-criticality` (we use $|m|$ to factor out the
finite-size sign-flipping of Exercise 3). Computed across temperature, $\chi$ stays tiny deep in the
ordered phase, then **peaks** sharply at $T_c$ ({numref}`fig-ising-criticality`) — and in the
thermodynamic limit the peak becomes a true **divergence**. This is precisely the breakdown [§5.9](grand-canonical-ensemble-equivalence.ipynb)
foretold: a critical point is where the relative fluctuations do *not* vanish as $1/\sqrt N$, where
a response function diverges, and where the system becomes correlated across its entire size. The
correlation length $\xi$ diverges, and the lattice fills with domains of *every* size — the shimmer
a real fluid shows at its critical point, called critical opalescence. The quantitative $\xi(T)$ and
the exponents that govern the divergence are MMM's; the concept — fluctuations gone macroscopic — is
the whole story.

**This exercise (worked).** Compute $\chi=N\operatorname{Var}(|m|)/kT$ (`numpy.var`) across a range
of temperatures and confirm it peaks near $T_c$, far above its deep-ordered value; show a snapshot
at $T_c$ with domains of every size.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    abs(peak_T - T_C) < 0.25 and chi.max() > 8 * chi[0],
    "the susceptibility — a fluctuation–response function (§5.9) — peaks at the critical point, where fluctuations stop vanishing as 1/√N",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Universality and the lattice-gas mapping

We arrive at the idea that makes the Ising model matter far beyond magnetism. Why should a toy of
$\pm1$ spins tell us anything about real materials? Because of what happens to *detail* at a
critical point. There the correlation length is enormous, so the system looks statistically the
same whether you squint or zoom — it is **scale invariant** — and under that magnification the
microscopic specifics (what the constituents are, the precise interaction) blur away. What survives
is only the coarsest information: the dimensionality of space and the symmetry of the order
parameter. Systems that agree on *those* fall into the same **universality class** and exhibit
*identical* critical behaviour, with the same critical exponents, no matter how different they look
up close.

The cleanest example is an exact change of variables {eq}`eq-universality`: relabel a spin
$\sigma_i=\pm1$ as an occupation $n_i=(1+\sigma_i)/2\in\{0,1\}$ — "up" becomes "an atom here", "down"
becomes "an empty site" — and the Ising magnet becomes a **lattice gas**. Its phase transition is
then the **liquid–vapour critical point** of a fluid; the magnetization maps to the density
difference between liquid and gas. The same algebra describes a binary **alloy** ordering, and the
same Ising-type model even reaches associative-memory neural networks. So the curve along which a
magnet loses its magnetization and the curve along which a fluid reaches its critical point are, near
the critical point, governed by *one* model. This is why a stripped-down toy explains real, unrelated
experiments — and why the Ising model is one of the most important models in physics. (The deep
*explanation*, the renormalization group, is the horizon beyond this course.)

**This exercise (worked).** Verify the magnet-to-fluid dictionary is an exact, invertible change of
variables — $\sigma\leftrightarrow n=(1+\sigma)/2$ — so the Ising magnet and the lattice gas are
literally the same model in two languages.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    np.array_equal(spins_back, spins) and set(occupation.tolist()) == {0, 1},
    "the Ising-to-lattice-gas map σ ↔ n=(1+σ)/2 is an exact change of variables — a magnet and a fluid are the same model (universality)",
)

## Exercise 6 — The transition from the other side: a quench

To feel the order *appear*, we approach the transition the way nature usually does — by cooling. We
take a hot, disordered configuration and suddenly drop the temperature below $T_c$ (a **quench**),
then watch ({numref}`fig-ising-quench`). At first the lattice is a fine speckle; almost immediately
small aligned patches form, and then the domains **coarsen** — small ones are eaten by their
neighbours, walls straighten and contract, and a single sign gradually takes over the lattice. Which
sign wins is decided by the chance imbalance in the early fluctuations, and once a large domain has
formed it is, by the ergodicity-breaking argument of Exercise 3, effectively frozen — the order is
"locked in." This is how a magnet cooled through its Curie point acquires a permanent magnetization,
and how, more generally, spontaneous order crystallizes out of a symmetric disordered state as a
system is taken across a phase transition. (How fast the domains coarsen, and the dynamics of the
approach to equilibrium, is the non-equilibrium physics that [§5.11](taste-of-nonequilibrium.ipynb) begins.)

**This exercise (student).** Quench a disordered lattice to $T<T_c$, record snapshots as the domains
coarsen, and confirm the final state is magnetized — spontaneous order frozen in by cooling. The
animation shows the coarsening.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    abs(quench_m[0]) < 0.2 and abs(quench_m[-1]) > 0.6,
    "a quench below T_c produces a magnetized state via domain coarsening — cooling through T_c freezes in spontaneous order",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Emergence: the whole volume in one model

Stand back and see what just happened, because it is the lesson the entire volume was built to
deliver. The Ising model is where everything converged. The microstate counting of [§5.1](counting.ipynb) and the
Boltzmann weight of [§5.4](microstates-entropy-temperature.ipynb) *defined* it. The Metropolis algorithm of [§5.8](partition-function.ipynb) *sampled* it. The
fluctuation–response relations of [§5.9](grand-canonical-ensemble-equivalence.ipynb) gave its *susceptibility*, and the ergodicity of [§5.5](ergodicity.ipynb) returned,
inverted, as the ergodicity *breaking* that makes its symmetry breaking real. And out of those
assembled parts came something that was in none of them: a sharp phase transition, spontaneous order
from a symmetric law, and the universality that lets a toy of $\pm1$ spins speak for real fluids and
alloys. **Emergence** — qualitatively new collective behaviour arising from simple local rules — is
the deepest lesson of statistical mechanics, and the Ising model shows it required *all* of the
machinery, working together.

**This exercise (synthesis).** No new computation: the convergence is the result. We began this
volume by counting microstates, and we end it by watching order create itself. Nothing in a single
spin knows what a phase transition is, yet together the spins break a symmetry their own law
respects. That gap — between the local rule and the collective fact — is where physics becomes deep.
For the quantitative craft (exact solutions, finite-size scaling, the critical exponents) the road
leads to the Materials Modelling course; for how a system *reaches* these equilibrium states at all,
and the arrow of time that points the way, 5.11 begins the non-equilibrium story.

## Notebook summary

The Ising model is the conceptual climax of the volume: collective order emerging from a symmetric
local rule. We pursued its *meaning* (the quantitative apparatus is the Materials Modelling
course's).

- **The model and its symmetry** {eq}`eq-ising`: spins $\pm1$ preferring to align, with a
  $\mathbb{Z}_2$ symmetry making "all up" and "all down" degenerate.
- **Spontaneous symmetry breaking** {eq}`eq-ssb`: below $T_c\approx2.269$ the system magnetizes,
  choosing one of two equal directions — the *law* symmetric, the *state* not (verified: all-up and
  all-down starts give $\pm m_0$).
- **Ergodicity breaking** {eq}`eq-ergodicity-breaking`: a *finite* magnet's signed $m$ flips sign
  (its time average is zero), and the flip rate collapses with size ($L=6$ flips freely, $L=16$
  barely) — true symmetry breaking exists only as $N\to\infty$, by trapping in one sector, the
  counterpart to the ergodicity of [§5.5](ergodicity.ipynb).
- **Criticality** {eq}`eq-ising-criticality`: the susceptibility $\chi=N\operatorname{Var}(|m|)/kT$ (a [§5.9](grand-canonical-ensemble-equivalence.ipynb)
  fluctuation–response function) peaks at $T_c$ (verified $\sim20$ vs $\sim0.1$ deep ordered) and
  diverges in the limit — fluctuations gone macroscopic, the correlation length diverging.
- **Universality** {eq}`eq-universality`: scale invariance at criticality washes out microscopic
  detail, so the magnet, the lattice gas (the liquid–vapour critical point, via $\sigma\leftrightarrow
  n=(1+\sigma)/2$), the binary alloy, and more share one universality class.

Emergence — new collective behaviour from local rules — needed the whole volume, and the Ising model
is where it all came together.

## Outlook

- **The quantitative Ising model — Materials Modelling (MMM).** The exact 1-D transfer matrix,
  Onsager's 2-D solution, finite-size scaling, the critical exponents, the correlation length
  $\xi(T)$, cluster algorithms, the Binder cumulant — the craft of extracting numbers, developed in
  full in MMM. This is the bidirectional link: ECP for the concepts, MMM for the computation.
- **The renormalization group.** The explanation of universality and scale invariance — why
  microscopic detail washes out at criticality — the horizon beyond both courses.
- **A taste of non-equilibrium (5.11).** How a system *reaches* equilibrium at all, the dynamics of
  coarsening and relaxation, and the arrow of time behind the equilibrium states this volume
  described.
- **The quantum Ising model (Volume VII).** The transverse-field Ising model and its quantum phase
  transition — order destroyed by quantum, not thermal, fluctuations — once Volume VI builds the
  quantum mechanics.
- **Cross-reference** [§5.5](ergodicity.ipynb) (ergodicity), [§5.8](partition-function.ipynb) (Metropolis), [§5.9](grand-canonical-ensemble-equivalence.ipynb) (the susceptibility fluctuation, the
  criticality bridge), [§5.1](counting.ipynb)/[§5.4](microstates-entropy-temperature.ipynb) (counting and the Boltzmann weight).

In [ ]:
from ecp.style import footer

footer()